# Chapter 1: Factor Graphs for SLAM — 基礎編

**SLAM Handbook Chapter 1** (Dellaert, Kaess, Barfoot) の内容をPythonで実装しながら学びます。

## このNotebookで学ぶこと

1. **Factor Graphとは何か** — SLAMの問題構造を可視化するグラフィカルモデル
2. **Toy Example** — 3ポーズ・2ランドマークの簡単なSLAM問題
3. **MAP推論と最小二乗** — 確率的推論が最適化問題に帰着する仕組み
4. **1D SLAMの実装** — NumPyで最小二乗を解いてみる

### 参考
- SLAM Handbook, Chapter 1: *Factor Graphs for SLAM* (Frank Dellaert, Michael Kaess, Timothy Barfoot)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import least_squares

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1.1 Factor Graphとは？

SLAMでは、ロボットの **ポーズ（位置・姿勢）** と **環境中のランドマーク** を同時に推定します。

この問題は **Factor Graph** というグラフィカルモデルで表現できます：

- **変数ノード（○）**: 推定したい未知数（ポーズ $\mathbf{p}_i$、ランドマーク $\boldsymbol{\ell}_j$）
- **ファクターノード（■）**: 変数間の制約（計測やオドメトリ）

### Toy Example (SLAM Handbook Fig. 1.1, 1.3)

ロボットが3つのポーズ $\mathbf{p}_1, \mathbf{p}_2, \mathbf{p}_3$ を移動しながら、2つのランドマーク $\boldsymbol{\ell}_1, \boldsymbol{\ell}_2$ を観測する状況を考えます。

In [ ]:
def draw_factor_graph():
    """SLAM Handbook Fig. 1.3 に対応するFactor Graphを描画"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    # 変数ノードの位置
    vars_pos = {
        'p1': (1, 0), 'p2': (4, 0), 'p3': (7, 0),
        'ℓ1': (3, 3), 'ℓ2': (6, 3),
    }

    # ファクターの定義: (位置, 接続する変数のリスト, ラベル)
    factors = [
        ((0, 0),    ['p1'],           'prior\n$\\phi_6(\\mathbf{p}_1)$'),
        ((2.5, 0),  ['p1', 'p2'],     'odom\n$\\phi_2(\\mathbf{p}_2,\\mathbf{p}_1)$'),
        ((5.5, 0),  ['p2', 'p3'],     'odom\n$\\phi_3(\\mathbf{p}_3,\\mathbf{p}_2)$'),
        ((3, 4),    ['ℓ1'],           'prior\n$\\phi_4(\\boldsymbol{\\ell}_1)$'),
        ((6, 4),    ['ℓ2'],           'prior\n$\\phi_5(\\boldsymbol{\\ell}_2)$'),
        ((1.5, 1.5),['p1', 'ℓ1'],    'obs\n$\\phi_7$'),
        ((3.5, 1.5),['p2', 'ℓ1'],    'obs\n$\\phi_8$'),
        ((6.5, 1.5),['p3', 'ℓ2'],    'obs\n$\\phi_9$'),
    ]

    # エッジを描画
    for (fx, fy), connected, _ in factors:
        for var_name in connected:
            vx, vy = vars_pos[var_name]
            ax.plot([fx, vx], [fy, vy], 'k-', linewidth=1.5, zorder=1)

    # 変数ノード（丸）を描画
    for name, (x, y) in vars_pos.items():
        circle = patches.Circle((x, y), 0.35, fill=True, facecolor='lightblue',
                                edgecolor='black', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, f'${name}$' if name[0] != '\\' else f'${name}$',
                ha='center', va='center', fontsize=13, fontweight='bold', zorder=4)

    # ファクターノード（四角）を描画
    for (fx, fy), _, label in factors:
        rect = patches.Rectangle((fx - 0.15, fy - 0.15), 0.3, 0.3,
                                 fill=True, facecolor='black', zorder=3)
        ax.add_patch(rect)
        # ラベルをファクターの横に表示
        ax.text(fx, fy - 0.6, label, ha='center', va='top', fontsize=8,
                color='darkred', zorder=4)

    ax.set_xlim(-1, 9)
    ax.set_ylim(-1.5, 5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Factor Graph: 3 Poses + 2 Landmarks (SLAM Handbook Fig. 1.3)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

draw_factor_graph()

## 1.2 MAP推論と最小二乗

### 事後確率の因数分解

ベイズの定理により、観測 $\mathbf{z}$ が与えられたときの状態 $\mathbf{x}$ の事後確率は：

$$p(\mathbf{x}|\mathbf{z}) \propto p(\mathbf{z}|\mathbf{x}) \, p(\mathbf{x})$$

Factor Graphでは、この事後確率をファクターの積として表現します：

$$\phi(\mathbf{x}) = \prod_i \phi_i(\mathbf{x}_i)$$

### ガウス分布の仮定

各ファクターがガウス分布に比例すると仮定すると（SLAM Handbook Eq. 1.17）：

$$\phi_i(\mathbf{x}_i) \propto \exp\left(-\frac{1}{2} \|\mathbf{z}_i - \mathbf{h}_i(\mathbf{x}_i)\|^2_{\boldsymbol{\Sigma}_i}\right)$$

ここで $\mathbf{h}_i$ は計測予測関数、$\boldsymbol{\Sigma}_i$ は計測ノイズの共分散行列です。

### MAP推定 = 非線形最小二乗

対数をとって符号を反転すると、**MAP推定は非線形最小二乗問題に帰着**します（SLAM Handbook Eq. 1.18）：

$$\mathbf{x}^{\text{MAP}} = \arg\min_{\mathbf{x}} \sum_i \|\mathbf{z}_i - \mathbf{h}_i(\mathbf{x}_i)\|^2_{\boldsymbol{\Sigma}_i}$$

ここで $\|\mathbf{e}\|^2_{\boldsymbol{\Sigma}} = \mathbf{e}^\top \boldsymbol{\Sigma}^{-1} \mathbf{e}$ はマハラノビス距離の2乗です。

## 1.3 実装: 1D SLAM問題

まず最もシンプルな **1次元のSLAM問題** を解いてみましょう。

### 問題設定

- ロボットは1次元の数直線上を移動
- 3つのポーズ $x_1, x_2, x_3$ を推定
- 2つのランドマーク $l_1, l_2$ を推定

### ファクター（制約）

| ファクター | 種類 | 意味 |
|-----------|------|------|
| $\phi_1(x_1)$ | Prior | $x_1$ の初期位置の事前情報 |
| $\phi_2(x_1, x_2)$ | Odometry | $x_1$ から $x_2$ への移動量 |
| $\phi_3(x_2, x_3)$ | Odometry | $x_2$ から $x_3$ への移動量 |
| $\phi_4(x_1, l_1)$ | Observation | $x_1$ から $l_1$ までの距離観測 |
| $\phi_5(x_2, l_1)$ | Observation | $x_2$ から $l_1$ までの距離観測 |
| $\phi_6(x_3, l_2)$ | Observation | $x_3$ から $l_2$ までの距離観測 |

In [ ]:
# =============================================================
# Ground Truth（真値）
# =============================================================
# ロボットのポーズ（1D位置）
x1_true, x2_true, x3_true = 0.0, 2.0, 4.0
# ランドマークの位置
l1_true, l2_true = 3.0, 7.0

print("=== Ground Truth ===")
print(f"Poses:     x1={x1_true}, x2={x2_true}, x3={x3_true}")
print(f"Landmarks: l1={l1_true}, l2={l2_true}")

In [ ]:
# =============================================================
# ノイズ付き観測を生成
# =============================================================
np.random.seed(42)

# ノイズの標準偏差
sigma_prior = 0.1     # 事前情報のノイズ
sigma_odom  = 0.5     # オドメトリのノイズ
sigma_obs   = 0.3     # ランドマーク観測のノイズ

# 事前情報: x1の位置
z_prior = x1_true + np.random.normal(0, sigma_prior)

# オドメトリ計測: ポーズ間の相対移動量
z_odom_12 = (x2_true - x1_true) + np.random.normal(0, sigma_odom)
z_odom_23 = (x3_true - x2_true) + np.random.normal(0, sigma_odom)

# ランドマーク観測: ポーズからランドマークまでの距離
z_obs_x1_l1 = (l1_true - x1_true) + np.random.normal(0, sigma_obs)
z_obs_x2_l1 = (l1_true - x2_true) + np.random.normal(0, sigma_obs)
z_obs_x3_l2 = (l2_true - x3_true) + np.random.normal(0, sigma_obs)

print("=== ノイズ付き観測 ===")
print(f"Prior:       z_prior     = {z_prior:.4f}  (真値: {x1_true})")
print(f"Odometry:    z_odom_12   = {z_odom_12:.4f}  (真値: {x2_true - x1_true})")
print(f"             z_odom_23   = {z_odom_23:.4f}  (真値: {x3_true - x2_true})")
print(f"Observation: z_obs_x1_l1 = {z_obs_x1_l1:.4f}  (真値: {l1_true - x1_true})")
print(f"             z_obs_x2_l1 = {z_obs_x2_l1:.4f}  (真値: {l1_true - x2_true})")
print(f"             z_obs_x3_l2 = {z_obs_x3_l2:.4f}  (真値: {l2_true - x3_true})")

### 方法1: 線形最小二乗（正規方程式）

1D SLAMでは全ての計測関数 $\mathbf{h}_i$ が線形なので、問題は **線形最小二乗** $\|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2$ の形に書けます。

状態ベクトル: $\mathbf{x} = [x_1, x_2, x_3, l_1, l_2]^\top$

各ファクターの残差 $e_i = \frac{z_i - h_i(\mathbf{x})}{\sigma_i}$（whitened residual）を行列形式にまとめます：

$$\mathbf{A}\mathbf{x} = \mathbf{b}$$

正規方程式 $\mathbf{A}^\top\mathbf{A} \, \mathbf{x}^* = \mathbf{A}^\top\mathbf{b}$ を解けばMAP推定値が得られます。

In [ ]:
# =============================================================
# 線形最小二乗で解く
# =============================================================
# 状態ベクトル: x = [x1, x2, x3, l1, l2]
# インデックス:      0    1    2   3   4

# Whitened Jacobian行列 A と観測ベクトル b を構築
# 各行が1つのファクターに対応
# e_i = (z_i - h_i(x)) / sigma_i  →  A_i @ x = b_i

# ファクター1: Prior on x1 → x1 / sigma_prior = z_prior / sigma_prior
# ファクター2: Odom x1→x2 → (x2 - x1) / sigma_odom = z_odom_12 / sigma_odom
# ファクター3: Odom x2→x3 → (x3 - x2) / sigma_odom = z_odom_23 / sigma_odom
# ファクター4: Obs x1→l1  → (l1 - x1) / sigma_obs = z_obs_x1_l1 / sigma_obs
# ファクター5: Obs x2→l1  → (l1 - x2) / sigma_obs = z_obs_x2_l1 / sigma_obs
# ファクター6: Obs x3→l2  → (l2 - x3) / sigma_obs = z_obs_x3_l2 / sigma_obs

n_vars = 5  # x1, x2, x3, l1, l2
n_factors = 6

A = np.zeros((n_factors, n_vars))
b = np.zeros(n_factors)

# Factor 1: x1 = z_prior  (prior)
A[0, 0] = 1.0 / sigma_prior
b[0] = z_prior / sigma_prior

# Factor 2: x2 - x1 = z_odom_12  (odometry)
A[1, 0] = -1.0 / sigma_odom
A[1, 1] =  1.0 / sigma_odom
b[1] = z_odom_12 / sigma_odom

# Factor 3: x3 - x2 = z_odom_23  (odometry)
A[2, 1] = -1.0 / sigma_odom
A[2, 2] =  1.0 / sigma_odom
b[2] = z_odom_23 / sigma_odom

# Factor 4: l1 - x1 = z_obs_x1_l1  (observation)
A[3, 0] = -1.0 / sigma_obs
A[3, 3] =  1.0 / sigma_obs
b[3] = z_obs_x1_l1 / sigma_obs

# Factor 5: l1 - x2 = z_obs_x2_l1  (observation)
A[4, 1] = -1.0 / sigma_obs
A[4, 3] =  1.0 / sigma_obs
b[4] = z_obs_x2_l1 / sigma_obs

# Factor 6: l2 - x3 = z_obs_x3_l2  (observation)
A[5, 2] = -1.0 / sigma_obs
A[5, 4] =  1.0 / sigma_obs
b[5] = z_obs_x3_l2 / sigma_obs

print("Whitened Jacobian A:")
print(A)
print(f"\nWhitened observation b: {b}")

In [ ]:
# =============================================================
# 正規方程式で解く: (A^T A) x* = A^T b
# =============================================================
ATA = A.T @ A
ATb = A.T @ b
x_normal = np.linalg.solve(ATA, ATb)

# QR分解でも解く（数値的に安定）
x_qr, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

print("=== MAP推定結果 ===")
labels = ['x1', 'x2', 'x3', 'l1', 'l2']
truths = [x1_true, x2_true, x3_true, l1_true, l2_true]

print(f"{'変数':>4s}  {'真値':>8s}  {'正規方程式':>8s}  {'QR分解':>8s}  {'誤差':>8s}")
print("-" * 50)
for i, (label, truth) in enumerate(zip(labels, truths)):
    print(f"{label:>4s}  {truth:8.4f}  {x_normal[i]:8.4f}  {x_qr[i]:8.4f}  {abs(x_normal[i]-truth):8.4f}")

### 推定の不確かさ（共分散行列）

情報行列 $\boldsymbol{\Lambda} = \mathbf{A}^\top\mathbf{A}$ の逆行列が推定値の共分散行列です：

$$\boldsymbol{\Sigma} = (\mathbf{A}^\top\mathbf{A})^{-1}$$

対角成分の平方根が各変数の推定標準偏差になります。

In [ ]:
# =============================================================
# 共分散行列（不確かさ）
# =============================================================
cov = np.linalg.inv(ATA)

print("共分散行列 Σ = (A^T A)^{-1}:")
print(cov)
print(f"\n各変数の推定標準偏差 (1σ):")
for i, label in enumerate(labels):
    print(f"  {label}: σ = {np.sqrt(cov[i, i]):.4f}")

## 1.4 結果の可視化

推定結果を真値と比較し、不確かさ（1σ区間）も表示します。

In [ ]:
# =============================================================
# 推定結果の可視化
# =============================================================
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

sigmas = np.sqrt(np.diag(cov))
colors = ['#2196F3', '#2196F3', '#2196F3', '#F44336', '#F44336']
markers = ['o', 'o', 'o', '*', '*']

for i, (label, truth, est, sig, color) in enumerate(
        zip(labels, truths, x_normal, sigmas, colors)):
    # 推定値と1σ区間
    ax.errorbar(est, 0.5, xerr=2*sig, fmt='o', color=color,
                markersize=10, capsize=5, capthick=2, linewidth=2,
                label=f'{label} est' if i < 1 or i == 3 else None)
    # 真値
    ax.plot(truth, 0.5, 'x', color='black', markersize=12, markeredgewidth=3)
    # ラベル
    ax.annotate(label, (est, 0.5), textcoords='offset points',
                xytext=(0, 20), ha='center', fontsize=12, fontweight='bold',
                color=color)

# オドメトリだけの推定（比較用）
x_odom = [z_prior, z_prior + z_odom_12, z_prior + z_odom_12 + z_odom_23]
for i, (xo, label) in enumerate(zip(x_odom, ['x1', 'x2', 'x3'])):
    ax.plot(xo, 0.2, 's', color='gray', markersize=8,
            label='odom only' if i == 0 else None)

ax.axhline(y=0.2, color='gray', linestyle='--', alpha=0.3)
ax.axhline(y=0.5, color='blue', linestyle='--', alpha=0.3)

ax.set_xlabel('Position', fontsize=14)
ax.set_yticks([0.2, 0.5])
ax.set_yticklabels(['Odometry only', 'MAP estimate'])
ax.set_ylim(-0.1, 1.0)
ax.legend(loc='upper left')
ax.set_title('1D SLAM: MAP推定 vs オドメトリのみ vs 真値(×)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.5 情報行列のスパース性

SLAM Handbook Section 1.5 の重要なポイント：**情報行列 $\mathbf{A}^\top\mathbf{A}$ はスパース**です。

これはFactor Graphの構造を反映しています。変数 $i$ と $j$ が同じファクターに接続されている場合のみ、情報行列の $(i, j)$ 要素が非ゼロになります。

In [ ]:
# =============================================================
# 情報行列のスパース構造を可視化
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Jacobian A のスパースパターン
axes[0].spy(A, markersize=15, color='steelblue')
axes[0].set_title('Jacobian A\n(各行=1ファクター)', fontsize=12)
axes[0].set_xticks(range(n_vars))
axes[0].set_xticklabels(labels)
axes[0].set_yticks(range(n_factors))
axes[0].set_yticklabels(['prior', 'odom12', 'odom23', 'obs_l1', 'obs_l1', 'obs_l2'],
                        fontsize=9)

# 情報行列 A^T A のスパースパターン
axes[1].spy(ATA, markersize=15, color='steelblue')
axes[1].set_title('Information Matrix\n$\\mathbf{A}^\\top\\mathbf{A}$', fontsize=12)
axes[1].set_xticks(range(n_vars))
axes[1].set_xticklabels(labels)
axes[1].set_yticks(range(n_vars))
axes[1].set_yticklabels(labels)

# 情報行列の値をヒートマップで表示
im = axes[2].imshow(np.abs(ATA), cmap='Blues', aspect='equal')
axes[2].set_title('|Information Matrix|\n(値の大きさ)', fontsize=12)
axes[2].set_xticks(range(n_vars))
axes[2].set_xticklabels(labels)
axes[2].set_yticks(range(n_vars))
axes[2].set_yticklabels(labels)
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

# スパース率を計算
nnz = np.count_nonzero(ATA)
total = ATA.size
print(f"情報行列のサイズ: {ATA.shape}")
print(f"非ゼロ要素数: {nnz} / {total} ({nnz/total*100:.0f}%)")
print(f"→ 問題が大きくなるほど、スパース率が高くなり効率的な解法が重要になります")

## 1.6 方法2: 非線形最小二乗（scipy.optimize）

1D問題では線形で解けましたが、実際のSLAMでは計測関数 $\mathbf{h}_i$ が **非線形** です（例: bearing計測では $\text{atan2}$ が入る）。

ここでは `scipy.optimize.least_squares` を使い、**非線形最小二乗ソルバー**で同じ問題を解いてみます。これはGauss-Newton法やLevenberg-Marquardt法の内部実装を持っています。

In [ ]:
# =============================================================
# 非線形最小二乗で解く（scipy.optimize.least_squares）
# =============================================================

def residuals(x):
    """各ファクターのwhitened residualを返す
    
    x = [x1, x2, x3, l1, l2]
    """
    x1, x2, x3, l1, l2 = x
    return np.array([
        (x1 - z_prior) / sigma_prior,          # Prior on x1
        (x2 - x1 - z_odom_12) / sigma_odom,    # Odometry x1→x2
        (x3 - x2 - z_odom_23) / sigma_odom,    # Odometry x2→x3
        (l1 - x1 - z_obs_x1_l1) / sigma_obs,   # Observation x1→l1
        (l1 - x2 - z_obs_x2_l1) / sigma_obs,   # Observation x2→l1
        (l2 - x3 - z_obs_x3_l2) / sigma_obs,   # Observation x3→l2
    ])

# 初期値（適当にずらす）
x0 = np.array([0.0, 0.0, 0.0, 0.0, 0.0])

# Levenberg-Marquardt法で解く
result = least_squares(residuals, x0, method='lm')

print("=== scipy.optimize.least_squares の結果 ===")
print(f"収束: {result.success}")
print(f"反復回数: {result.nfev}")
print(f"最終コスト: {result.cost:.6f}")
print()
print(f"{'変数':>4s}  {'真値':>8s}  {'NLS推定':>8s}  {'線形LS推定':>8s}  {'差分':>8s}")
print("-" * 55)
for i, (label, truth) in enumerate(zip(labels, truths)):
    diff = abs(result.x[i] - x_normal[i])
    print(f"{label:>4s}  {truth:8.4f}  {result.x[i]:8.4f}  {x_normal[i]:8.4f}  {diff:8.2e}")

print(f"\n→ 線形問題なので、線形LSと非線形LSの結果は一致します")

## 1.7 演習問題

### 演習1: ノイズの影響
上のセルで `sigma_odom` を `0.1` や `2.0` に変えて再実行してみましょう。
- オドメトリの信頼度が変わると、推定結果と不確かさはどう変化しますか？

### 演習2: ループクロージャ
新しいファクター $\phi(x_3, l_1)$（ポーズ3からランドマーク1を観測）を追加してみましょう。
- `A` 行列に1行追加するだけです
- ループクロージャによって不確かさがどう減るか確認してください

### 演習3: 2Dへの拡張
ポーズを $(x, y)$ の2次元に拡張し、ランドマークまでの **距離** を観測するモデルを実装してみましょう。
- 計測関数 $h(\mathbf{p}, \boldsymbol{\ell}) = \|\boldsymbol{\ell} - \mathbf{p}\|$ は非線形になるため、Gauss-Newton法が必要です

## まとめ

- **Factor Graph** はSLAMの問題構造（変数間の依存関係）を表現するグラフィカルモデル
- ガウスノイズの仮定のもと、**MAP推定 = 最小二乗問題** に帰着する
- 情報行列 $\mathbf{A}^\top\mathbf{A}$ は **スパース** → 大規模SLAM問題でも効率的に解ける
- 次のNotebookでは、MAP推論とGauss-Newton法をさらに深掘りします

### 次のNotebook
→ `02_map_inference.ipynb`: MAP推論の詳細と非線形最適化手法